# LokaSense Inference Notebook

This notebook is the lightweight companion to the training notebook.
Train or refresh checkpoints from `training.ipynb`, then use this notebook to run local inference directly from cells.
It loads the saved signal and NER checkpoints and runs end-user inference directly from notebook cells.


## What This Notebook Does

- Loads the latest local `models/signal_base` checkpoint
- Loads the latest local `models/ner_base` checkpoint
- Runs signal classification on editable Indonesian example text
- Extracts NER spans from the same text so you can quickly inspect location and organization hits


In [1]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForSequenceClassification, AutoModelForTokenClassification, AutoTokenizer

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "models").exists(), "Please run this notebook from the repo root."
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SIGNAL_MODEL_DIR = REPO_ROOT / "models" / "signal_base"
NER_MODEL_DIR = REPO_ROOT / "models" / "ner_base"
SIGNAL_LABELS = [
    "NEUTRAL",
    "DEMAND_UNMET",
    "DEMAND_PRESENT",
    "SUPPLY_SIGNAL",
    "COMPETITION_HIGH",
    "COMPLAINT",
    "TREND",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert SIGNAL_MODEL_DIR.exists(), f"Missing signal model at {SIGNAL_MODEL_DIR}"
assert NER_MODEL_DIR.exists(), f"Missing NER model at {NER_MODEL_DIR}"

signal_tokenizer = AutoTokenizer.from_pretrained(str(SIGNAL_MODEL_DIR))
signal_model = AutoModelForSequenceClassification.from_pretrained(str(SIGNAL_MODEL_DIR)).to(device)
signal_model.eval()

ner_tokenizer = AutoTokenizer.from_pretrained(str(NER_MODEL_DIR))
ner_model = AutoModelForTokenClassification.from_pretrained(str(NER_MODEL_DIR)).to(device)
ner_model.eval()

display(
    pd.DataFrame(
        [
            {
                "signal_model": str(SIGNAL_MODEL_DIR),
                "ner_model": str(NER_MODEL_DIR),
                "device": str(device),
            }
        ]
    )
)


def regex_tokens(text: str):
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def predict_signal(texts):
    encoded = signal_tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    encoded = {key: value.to(device) for key, value in encoded.items()}
    with torch.no_grad():
        logits = signal_model(**encoded).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

    rows = []
    for text, prob in zip(texts, probs):
        ranked = np.argsort(prob)[::-1]
        rows.append(
            {
                "text": text,
                "prediction": SIGNAL_LABELS[int(ranked[0])],
                "confidence": round(float(prob[ranked[0]]), 4),
                "runner_up": SIGNAL_LABELS[int(ranked[1])],
                "runner_up_confidence": round(float(prob[ranked[1]]), 4),
                "third_choice": SIGNAL_LABELS[int(ranked[2])],
                "third_choice_confidence": round(float(prob[ranked[2]]), 4),
            }
        )
    return pd.DataFrame(rows)


def predict_ner_entities(text: str):
    tokens = regex_tokens(text)
    encoded = ner_tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=128, return_tensors="pt")
    word_ids = encoded.word_ids(batch_index=0)
    encoded = {key: value.to(device) for key, value in encoded.items()}
    with torch.no_grad():
        logits = ner_model(**encoded).logits
    pred_ids = torch.argmax(logits, dim=-1)[0].cpu().tolist()

    aligned = []
    previous_word_id = None
    for idx, word_id in enumerate(word_ids):
        if word_id is None or word_id == previous_word_id:
            previous_word_id = word_id
            continue
        aligned.append((tokens[word_id], ner_model.config.id2label[pred_ids[idx]]))
        previous_word_id = word_id

    entities = []
    current_tokens = []
    current_label = None
    for token, tag in aligned:
        if tag.startswith("B-"):
            if current_tokens:
                entities.append({"entity": " ".join(current_tokens), "label": current_label})
            current_tokens = [token]
            current_label = tag[2:]
        elif tag.startswith("I-") and current_tokens and current_label == tag[2:]:
            current_tokens.append(token)
        else:
            if current_tokens:
                entities.append({"entity": " ".join(current_tokens), "label": current_label})
            current_tokens = []
            current_label = None
    if current_tokens:
        entities.append({"entity": " ".join(current_tokens), "label": current_label})
    return pd.DataFrame(entities or [{"entity": "", "label": "NO_ENTITY"}])


/home/parasite/Project/Competition/UGM_HACKATHON/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,signal_model,ner_model,device
0,/home/parasite/Project/Competition/UGM_HACKATH...,/home/parasite/Project/Competition/UGM_HACKATH...,cuda


The setup cell loads both models into memory and binds them to the active device.
If a checkpoint is missing, it will fail early with a clear path so you know which training artifact still needs to be generated.


In [2]:
inference_inputs = pd.DataFrame(
    [
        {
            "text": "tolong buka kedai kopi murah di lowokwaru malang, anak kos butuh tempat nongkrong dekat kampus",
            "city": "Malang",
            "area_hint": "Lowokwaru",
            "business_hint": "kedai kopi",
        },
        {
            "text": "cafe di wonokromo sekarang saingannya banyak banget, hampir tiap jalan ada tempat baru",
            "city": "Surabaya",
            "area_hint": "Wonokromo",
            "business_hint": "cafe",
        },
        {
            "text": "bakso ini lagi viral di kotagede jogja, rame terus dan reviewnya bagus",
            "city": "Yogyakarta",
            "area_hint": "Kotagede",
            "business_hint": "bakso",
        },
        {
            "text": "pelayanan warung makan di cicendo mengecewakan, mahal dan rasanya biasa aja",
            "city": "Bandung",
            "area_hint": "Cicendo",
            "business_hint": "warung makan",
        },
    ]
)
display(inference_inputs)


,text,city,area_hint,business_hint
0,tolong buka kedai kopi murah di lowokwaru mala...,Malang,Lowokwaru,kedai kopi
1,cafe di wonokromo sekarang saingannya banyak b...,Surabaya,Wonokromo,cafe
2,"bakso ini lagi viral di kotagede jogja, rame t...",Yogyakarta,Kotagede,bakso
3,pelayanan warung makan di cicendo mengecewakan...,Bandung,Cicendo,warung makan


Edit the `inference_inputs` cell if you want to test your own Indonesian text.
Keeping `city`, `area_hint`, and `business_hint` nearby also makes it easier to reuse these rows later for downstream scoring or demos.


In [3]:
signal_predictions = predict_signal(inference_inputs["text"].tolist())
display(pd.concat([inference_inputs, signal_predictions.drop(columns=["text"])], axis=1))


,text,city,area_hint,business_hint,prediction,confidence,runner_up,runner_up_confidence,third_choice,third_choice_confidence
0,tolong buka kedai kopi murah di lowokwaru mala...,Malang,Lowokwaru,kedai kopi,DEMAND_PRESENT,0.2628,SUPPLY_SIGNAL,0.2138,DEMAND_UNMET,0.2076
1,cafe di wonokromo sekarang saingannya banyak b...,Surabaya,Wonokromo,cafe,DEMAND_PRESENT,0.3075,DEMAND_UNMET,0.1987,NEUTRAL,0.1857
2,"bakso ini lagi viral di kotagede jogja, rame t...",Yogyakarta,Kotagede,bakso,DEMAND_PRESENT,0.3657,SUPPLY_SIGNAL,0.1843,DEMAND_UNMET,0.1685
3,pelayanan warung makan di cicendo mengecewakan...,Bandung,Cicendo,warung makan,DEMAND_PRESENT,0.3444,NEUTRAL,0.2938,DEMAND_UNMET,0.1369


The signal classifier output shows the top prediction and the next two alternatives.
That is usually more useful than a single hard label when you are debugging borderline texts like `DEMAND_UNMET` versus `DEMAND_PRESENT`.


In [4]:
ner_outputs = []
for _, row in inference_inputs.iterrows():
    entities = predict_ner_entities(row["text"]).to_dict(orient="records")
    ner_outputs.append({"text": row["text"], "entities": entities})

ner_outputs_df = pd.DataFrame(ner_outputs)
display(ner_outputs_df)


,text,entities
0,tolong buka kedai kopi murah di lowokwaru mala...,"[{'entity': 'kopi', 'label': 'ORG'}, {'entity'..."
1,cafe di wonokromo sekarang saingannya banyak b...,"[{'entity': 'wonokromo', 'label': 'LOC'}]"
2,"bakso ini lagi viral di kotagede jogja, rame t...","[{'entity': 'bakso', 'label': 'ORG'}, {'entity..."
3,pelayanan warung makan di cicendo mengecewakan...,"[{'entity': 'cicendo', 'label': 'LOC'}]"


The NER output is deliberately kept simple here: you get back the extracted entities and their labels per text.
That makes the notebook easy to use as a local demo surface without needing to run the full spatial aggregation pipeline.


In [5]:
batch_demo_path = REPO_ROOT / "data" / "social_media" / "tiktok_data.csv"
if batch_demo_path.exists():
    batch_demo = pd.read_csv(batch_demo_path).head(8).copy()
    batch_demo_predictions = predict_signal(batch_demo["text"].fillna("").astype(str).tolist())
    display(pd.concat([batch_demo[["text", "city", "area_hint", "business_hint"]], batch_demo_predictions.drop(columns=["text"])], axis=1))
else:
    print(f"No demo batch found at {batch_demo_path}")


,text,city,area_hint,business_hint,prediction,confidence,runner_up,runner_up_confidence,third_choice,third_choice_confidence
0,halo warga malang sekarang mie ayam keraton bu...,Malang,Lowokwaru,mie,DEMAND_PRESENT,0.3062,SUPPLY_SIGNAL,0.2322,DEMAND_UNMET,0.1518
1,sudah pada nyobain guys kuliner malang kuliner...,Malang,Lowokwaru,mie,SUPPLY_SIGNAL,0.3128,DEMAND_PRESENT,0.2385,DEMAND_UNMET,0.1743
2,jangan lebay tapi ini mala nya balance kuliner...,Malang,Lowokwaru,mie,DEMAND_PRESENT,0.2963,SUPPLY_SIGNAL,0.2157,DEMAND_UNMET,0.2064
3,mia ayam langganan siapa nih kuliner malang ku...,Malang,Lowokwaru,mie,DEMAND_PRESENT,0.2868,SUPPLY_SIGNAL,0.2584,DEMAND_UNMET,0.1653
4,cocok banget pas malang lagi ujan kulinermalan...,Malang,Lowokwaru,mie,SUPPLY_SIGNAL,0.3344,DEMAND_PRESENT,0.2324,DEMAND_UNMET,0.1518
5,menu ramyeon tray topping baru haus kali ini b...,Malang,Lowokwaru,mie,SUPPLY_SIGNAL,0.3643,DEMAND_PRESENT,0.1697,TREND,0.1506
6,"mie ayam porsi barbar, harga mulai 10 ribu saj...",Malang,Lowokwaru,mie,TREND,0.5607,SUPPLY_SIGNAL,0.1721,DEMAND_UNMET,0.1002
7,"mie marem jaya jl. pasar tawangmangu no.c 26, ...",Malang,Lowokwaru,mie,TREND,0.2189,SUPPLY_SIGNAL,0.2097,DEMAND_PRESENT,0.1939


This optional batch cell gives you a quick sanity check on real scraped text without retraining anything.
It is a nice first stop when you want to know whether the saved checkpoints are at least behaving plausibly on the current raw corpus.
